# Exercícios de Regressão Logística — Resolução

Resolução dos exercícios do **Guia de Estudos - Prova.md**, seguindo o mesmo padrão de código
usado na Aula 5 (função sigmoide e entropia cruzada implementadas com Numpy puro).

In [1]:
# =======================
# funções do modelo
# =======================
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


## Exercício 1 — Classificando clientes

Um banco usa $z = -3 + 0{,}8 \cdot \text{renda (em milhares)} - 0{,}5 \cdot \text{nº de dívidas ativas}$
para prever se aprova (1) ou não (0) um crédito.

| Cliente | Renda | Dívidas |
|---|---|---|
| P | 5 | 1 |
| Q | 3 | 0 |
| R | 4 | 3 |

**Pergunta:** calcule $z$, a probabilidade $\sigma(z)$ e a classificação final de cada cliente.

In [2]:
beta0, beta1, beta2 = -3, 0.8, -0.5  # coeficientes do modelo já treinado (fornecidos pelo exercício)


#### Obtendo os dados dos clientes — duas formas equivalentes

In [3]:
# Forma 1: dados digitados direto no código
nomes = ["P", "Q", "R"]
dados_clientes = np.array([
    [5, 1],   # renda, dívidas do cliente P
    [3, 0],   # renda, dívidas do cliente Q
    [4, 3],   # renda, dívidas do cliente R
])

print("nomes:", nomes)
print("dados_clientes:\n", dados_clientes)


nomes: ['P', 'Q', 'R']
dados_clientes:
 [[5 1]
 [3 0]
 [4 3]]


In [4]:
# Forma 2: carregando os MESMOS dados de um arquivo CSV
# arquivo "exercicio1_clientes.csv":
#   cliente,renda,dividas
#   P,5,1
#   Q,3,0
#   R,4,3
# usecols=(1, 2) pula a 1ª coluna (nome, é texto, não dá pra carregar com np.loadtxt)
dados_clientes = np.loadtxt("exercicio1_clientes.csv", delimiter=",", skiprows=1, usecols=(1, 2))
nomes = ["P", "Q", "R"]  # nomes continuam à parte, pois não são um dado numérico

print("nomes:", nomes)
print("dados_clientes:\n", dados_clientes)


nomes: ['P', 'Q', 'R']
dados_clientes:
 [[5. 1.]
 [3. 0.]
 [4. 3.]]


In [5]:
for nome, (renda, dividas) in zip(nomes, dados_clientes):
    z = beta0 + beta1 * renda + beta2 * dividas
    prob = sigmoid(z)
    classe = 1 if prob >= 0.5 else 0
    resultado_texto = "APROVADO" if classe == 1 else "NEGADO"
    print(f"Cliente {nome}: z={z:.2f} | probabilidade={prob:.4f} ({prob * 100:.1f}%) | crédito {resultado_texto}")


Cliente P: z=0.50 | probabilidade=0.6225 (62.2%) | crédito APROVADO
Cliente Q: z=-0.60 | probabilidade=0.3543 (35.4%) | crédito NEGADO
Cliente R: z=-1.30 | probabilidade=0.2142 (21.4%) | crédito NEGADO


### Respostas — Exercício 1

Resultados calculados na célula acima:

- **P:** $z=0{,}5 \to \sigma(0{,}5)\approx0{,}622$ (62,2%) → **crédito aprovado**
- **Q:** $z=-0{,}6 \to \sigma(-0{,}6)\approx0{,}354$ (35,4%) → **crédito negado**
- **R:** $z=-1{,}3 \to \sigma(-1{,}3)\approx0{,}214$ (21,4%) → **crédito negado**

## Exercício 2 — Comparando o custo (log loss) de dois modelos

Um modelo previu as probabilidades $p=[0{,}8,\ 0{,}3,\ 0{,}6]$ para 3 exemplos cujos valores reais
são $y=[1,\ 0,\ 1]$. Um segundo modelo previu $p=[0{,}95,\ 0{,}1,\ 0{,}9]$ para os mesmos $y$.

**Perguntas:** a) calcule o log loss de cada modelo; b) qual modelo é melhor? Por quê?

In [6]:
def compute_cost(y, h):
    """Calcula o custo (log loss / entropia cruzada binária)."""
    m = len(y)
    epsilon = 1e-8
    cost = -(1 / m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )
    return cost


#### Obtendo y e as probabilidades — duas formas equivalentes

In [7]:
# Forma 1: dados digitados direto no código
y = np.array([1, 0, 1])
p_modelo1 = np.array([0.8, 0.3, 0.6])
p_modelo2 = np.array([0.95, 0.1, 0.9])

print("y:", y)
print("p_modelo1:", p_modelo1)
print("p_modelo2:", p_modelo2)


y: [1 0 1]
p_modelo1: [0.8 0.3 0.6]
p_modelo2: [0.95 0.1  0.9 ]


In [8]:
# Forma 2: carregando os MESMOS dados de um arquivo CSV
# arquivo "exercicio2_probabilidades.csv":
#   y_real,p_modelo1,p_modelo2
#   1,0.8,0.95
#   0,0.3,0.1
#   1,0.6,0.9
dados = np.loadtxt("exercicio2_probabilidades.csv", delimiter=",", skiprows=1)
y = dados[:, 0]
p_modelo1 = dados[:, 1]
p_modelo2 = dados[:, 2]

print("y:", y)
print("p_modelo1:", p_modelo1)
print("p_modelo2:", p_modelo2)


y: [1. 0. 1.]
p_modelo1: [0.8 0.3 0.6]
p_modelo2: [0.95 0.1  0.9 ]


In [9]:
custo_modelo1 = compute_cost(y, p_modelo1)
custo_modelo2 = compute_cost(y, p_modelo2)

print(f"Custo (log loss) do Modelo 1: {custo_modelo1:.4f}")
print(f"Custo (log loss) do Modelo 2: {custo_modelo2:.4f}")

melhor_modelo = "Modelo 1" if custo_modelo1 < custo_modelo2 else "Modelo 2"
print(f"\n--> O {melhor_modelo} é melhor: quanto menor o custo, mais as probabilidades previstas")
print("    se aproximam dos valores reais.")


Custo (log loss) do Modelo 1: 0.3635
Custo (log loss) do Modelo 2: 0.0873

--> O Modelo 2 é melhor: quanto menor o custo, mais as probabilidades previstas
    se aproximam dos valores reais.


### Respostas — Exercício 2

**a)** Log loss do Modelo 1 ≈ **0,3635** (calculado na célula acima).

**b)** Log loss do Modelo 2 ≈ **0,0873** (calculado na célula acima).

**c)** O **Modelo 2** é melhor: seu log loss (≈0,087) é bem menor que o do Modelo 1 (≈0,364).
Quanto menor o custo, mais as probabilidades previstas se aproximam dos valores reais.

## Exercício 3 — Treinando um modelo do zero (igual ao notebook da aula)

Nos Exercícios 1 e 2 usamos um modelo **já treinado** (os $\beta$ vinham prontos no enunciado).
Aqui vamos fazer o que o notebook `regressao_logistica.ipynb` faz de verdade: **treinar os
próprios $\beta$** a partir de dados, usando gradiente descendente — os mesmos blocos (funções
do modelo, pré-processamento, treino, avaliação) na mesma ordem.

**Cenário:** uma academia quer prever se um aluno vai **renovar o plano** (1) ou não (0), a
partir de `meses_membro` (há quanto tempo é aluno) e `faltas_mes` (faltas no último mês):

| meses_membro | faltas_mes | renovou | meses_membro | faltas_mes | renovou |
|---|---|---|---|---|---|
| 1 | 5 | 0 | 4 | 2 | 1 |
| 2 | 4 | 0 | 5 | 1 | 1 |
| 1 | 6 | 0 | 6 | 0 | 1 |
| 3 | 3 | 0 | 4 | 3 | 1 |
| 2 | 5 | 0 | 7 | 1 | 1 |
| 3 | 4 | 0 | 8 | 0 | 1 |
| 2 | 6 | 0 | 5 | 2 | 1 |
| 1 | 4 | 0 | 6 | 1 | 1 |

16 alunos ao todo (8 que não renovaram, 8 que renovaram) — dataset pequeno de propósito, só para
o treino rodar rápido e dar pra acompanhar cada etapa.

#### Obtendo os dados — duas formas equivalentes

In [10]:
# Forma 1: dados digitados direto no código
X_features = np.array([
    [1, 5], [2, 4], [1, 6], [3, 3], [2, 5], [3, 4], [2, 6], [1, 4],  # não renovaram
    [4, 2], [5, 1], [6, 0], [4, 3], [7, 1], [8, 0], [5, 2], [6, 1],  # renovaram
])
y_completo = np.array([0,0,0,0,0,0,0,0, 1,1,1,1,1,1,1,1]).reshape(-1, 1)

print("X_features:\n", X_features)
print("y_completo:", y_completo.ravel())


X_features:
 [[1 5]
 [2 4]
 [1 6]
 [3 3]
 [2 5]
 [3 4]
 [2 6]
 [1 4]
 [4 2]
 [5 1]
 [6 0]
 [4 3]
 [7 1]
 [8 0]
 [5 2]
 [6 1]]
y_completo: [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]


In [11]:
# Forma 2: carregando os MESMOS dados de um arquivo CSV
# arquivo "exercicio3_academia.csv":
#   meses_membro,faltas_mes,renovou
#   1,5,0
#   2,4,0
#   ...
dados = np.loadtxt("exercicio3_academia.csv", delimiter=",", skiprows=1)
X_features = dados[:, :2]                      # colunas 0 e 1 = meses_membro, faltas_mes
y_completo = dados[:, 2].reshape(-1, 1)         # última coluna = renovou (y)

print("X_features:\n", X_features)
print("y_completo:", y_completo.ravel())


X_features:
 [[1. 5.]
 [2. 4.]
 [1. 6.]
 [3. 3.]
 [2. 5.]
 [3. 4.]
 [2. 6.]
 [1. 4.]
 [4. 2.]
 [5. 1.]
 [6. 0.]
 [4. 3.]
 [7. 1.]
 [8. 0.]
 [5. 2.]
 [6. 1.]]
y_completo: [0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1.]


#### Bloco 1 — Funções do modelo (sigmoid, custo e gradiente descendente)

In [12]:
# sigmoid: "espreme" o z (combinação linear) para uma probabilidade entre 0 e 1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# compute_cost: mede o quão "erradas" estão as previsões do modelo (log loss / entropia cruzada).
# ATENÇÃO: aqui a assinatura é diferente da do Exercício 2 (lá era compute_cost(y, h), recebendo
# as probabilidades já prontas). Aqui, igual ao notebook da aula, a função recebe X e beta e
# calcula h = sigmoid(X @ beta) internamente -- é assim que compute_cost aparece dentro do
# gradient_descent, pois a cada época o beta muda e h precisa ser recalculado.
def compute_cost(X, y, beta):
    m = len(y)                       # número de exemplos
    h = sigmoid(X @ beta)            # probabilidades previstas para TODOS os exemplos de uma vez
    epsilon = 1e-8                   # evita log(0), que quebraria a conta
    cost = -(1 / m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )
    return cost


# gradient_descent: ajusta o beta aos poucos, "descendo a ladeira" do custo.
def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)
    for i in range(epochs):                    # repete "epochs" vezes (cada repetição = 1 época)
        h = sigmoid(X @ beta)                  # 1. calcula as probabilidades com o beta atual
        gradient = (1 / m) * (X.T @ (h - y))   # 2. calcula a direção que mais AUMENTA o custo
        beta = beta - lr * gradient            # 3. anda na direção CONTRÁRIA (por isso o "-"),
                                                #    um passo de tamanho "lr" (taxa de aprendizado)
        if i % 200 == 0:                       # a cada 200 épocas, mostra o custo caindo
            print(f"Epoch {i} - Custo: {compute_cost(X, y, beta):.4f}")
    return beta


#### Bloco 2 — Pré-processamento (normalização e bias)

Diferente do notebook da aula (que usa dados do Titanic, com idades faltando), aqui os dados
já vêm completos -- não precisamos de um `fill_missing_age`. Mas **normalizar continua
necessário**: `meses_membro` vai de 1 a 8 e `faltas_mes` de 0 a 6, escalas parecidas neste caso,
mas normalizar também deixa os coeficientes finais comparáveis entre si (útil para interpretar
qual variável "pesa" mais).

In [13]:
# normalize: deixa cada coluna com média 0 e desvio padrão 1 (mesma fórmula do notebook da aula)
def normalize(X, mean, std):
    return (X - mean) / (std + 1e-8)


# add_bias: adiciona a coluna de 1's à esquerda, para representar o beta_0 (intercepto)
def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))


#### Bloco 3 — Separando treino e teste

In [14]:
# train_test_split: separa uma fatia dos dados para treinar e outra para testar depois --
# o modelo NUNCA vê os dados de teste durante o treino, para avaliarmos de forma honesta.
def train_test_split(X, y, test_size=0.25, seed=42):
    np.random.seed(seed)                        # seed fixa = embaralhamento sempre igual (reprodutível)
    indices = np.arange(len(X))
    np.random.shuffle(indices)                  # embaralha a ordem dos exemplos
    split = int(len(X) * (1 - test_size))        # ponto de corte (75% treino / 25% teste)
    train_idx, test_idx = indices[:split], indices[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = train_test_split(X_features, y_completo, test_size=0.25)
print("Tamanho do treino:", len(X_train), "exemplos")
print("Tamanho do teste:", len(X_test), "exemplos")


Tamanho do treino: 12 exemplos
Tamanho do teste: 4 exemplos


#### Bloco 4 — Treinando o modelo (gradiente descendente de verdade)

In [15]:
# Normaliza o treino usando a média/desvio DO PRÓPRIO TREINO
mean_train = np.mean(X_train, axis=0)
std_train = np.std(X_train, axis=0)
X_train_norm = normalize(X_train, mean_train, std_train)
X_train_final = add_bias(X_train_norm)

# beta começa todo zerado -- o modelo "não sabe nada" ainda, vai aprender treinando
beta = np.zeros((X_train_final.shape[1], 1))

# Treina de verdade: o custo deve CAIR a cada bloco de 200 épocas impresso
beta = gradient_descent(X_train_final, y_train, beta, lr=0.5, epochs=2000)

print("\nBeta final [intercepto, meses_membro, faltas_mes]:", beta.ravel())


Epoch 0 - Custo: 0.5273
Epoch 200 - Custo: 0.0354
Epoch 400 - Custo: 0.0223
Epoch 600 - Custo: 0.0166
Epoch 800 - Custo: 0.0134
Epoch 1000 - Custo: 0.0113
Epoch 1200 - Custo: 0.0097
Epoch 1400 - Custo: 0.0086
Epoch 1600 - Custo: 0.0077
Epoch 1800 - Custo: 0.0070

Beta final [intercepto, meses_membro, faltas_mes]: [ 1.443463    6.23645882 -6.60171377]


#### Bloco 5 — Avaliando no conjunto de teste

**Importante:** o teste precisa ser normalizado com a **mesma** média/desvio do treino (não
recalculamos em cima do teste) -- é assim que o modelo "vê" o teste no mesmo formato que
aprendeu. Essa é justamente a melhoria em relação ao detalhe que comentei sobre o notebook da
aula, que recalcula média/desvio direto no teste.

In [16]:
X_test_norm = normalize(X_test, mean_train, std_train)   # usa mean_train/std_train, não os do teste!
X_test_final = add_bias(X_test_norm)

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)          # limiar de decisão: 0.5

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

y_pred = predict(X_test_final, beta)
acc = accuracy(y_test, y_pred)

print("Previsões no teste:  ", y_pred.ravel())
print("Valores reais no teste:", y_test.ravel())
print(f"Acurácia no teste: {acc:.4f} ({acc * 100:.1f}%)")


Previsões no teste:   [1 1 1 0]
Valores reais no teste: [1. 1. 0. 0.]
Acurácia no teste: 0.7500 (75.0%)


#### Bloco 6 — Prevendo um aluno novo

In [17]:
# Aluno novo: 3 meses como membro, 2 faltas no mês -- precisa passar pelo MESMO pré-processamento
novo_aluno = np.array([[3, 2]])
novo_aluno_norm = normalize(novo_aluno, mean_train, std_train)
novo_aluno_final = add_bias(novo_aluno_norm)

pred_novo = predict(novo_aluno_final, beta)
resultado_texto = "RENOVA" if pred_novo[0][0] == 1 else "NÃO RENOVA"
print(f"Aluno novo (3 meses, 2 faltas) -> previsão: {resultado_texto}")


Aluno novo (3 meses, 2 faltas) -> previsão: RENOVA


### Explicação dos resultados — Exercício 3

- **O custo caiu** de ≈0,53 (na época 0, quando beta ainda era zero, ou seja, "chute aleatório")
  para menos de 0,01 depois de 2000 épocas -- prova de que o gradiente descendente está mesmo
  aprendendo, e não travado.
- **Beta final** ficou aproximadamente `[1,44, 6,24, -6,60]` (intercepto, meses_membro, faltas_mes).
  Como os dados foram normalizados, dá pra comparar o tamanho dos coeficientes diretamente: o
  peso de `faltas_mes` (-6,60) é ligeiramente maior em módulo que o de `meses_membro` (6,24) —
  ou seja, faltar pesa um pouco mais contra a renovação do que o tempo de casa pesa a favor.
  Os sinais fazem sentido: mais meses de contrato **aumenta** a chance de renovar (+), mais
  faltas **reduz** a chance de renovar (-).
- **Acurácia no teste ≈ 75%** (acertou 3 de 4 exemplos que nunca tinha visto durante o treino) —
  o único erro foi um caso "de fronteira", parecido com o que discutimos no Exercício 2 da
  Aula 6 sobre a diferença entre acertar tudo e generalizar bem.
- O **aluno novo** (3 meses, 2 faltas -- um caso intermediário no meio dos dois grupos) foi
  classificado como **"renova"**, mostrando o modelo aplicando o que aprendeu a um caso que
  nunca viu.